<a href="https://colab.research.google.com/github/SampMark/Machine-Learning/blob/main/Log_Loss_Cross_Entropy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **A Função de Custo de Entropia Cruzada (Cross-Entropy Loss) $ J(\theta) $**

A função de custo para cálculo da entropia cruzada (Cross-Entropy Loss) é usada em **aprendizado de máquina** em modelos de **classificação multiclasse**, como na [**Regressão Softmax**](https://github.com/SampMark/Machine-Learning/blob/main/Softmax_Multinomial_Regression.ipynb) e incorpora **regularização**, o que a torna ideal para problemas de classificação com múltiplas classes.  Ou seja, ela combina **entropia cruzada multiclasse** com **regularização $L2$**, sendo amplamente utilizada em problemas de classificação com múltiplas classes.

A função de custo de entropia cruzada com regularização L2 é fundamental em modelagens com redes neurais e aprendizado profundo, sendo usada como base para a classificação em redes neurais convolucionais (CNN) e modelos de Processamento de Lingugem Natural (NLP) como Transformers. Em linhas gerais, tem por objetivo medir a discrepância entre as probabilidades previstas pelo modelo e as classes reais. Minimizar a função melhora a precisão do modelo e evita sobreajuste.

## **Definição matemática**
A função de custo $J(\theta)$ é uma combinação convexa de:
1. **Função de Perda (Entropia Cruzada):** para medir o ajuste aos dados.
2. **Regularização \(L_2\):** para controlar a complexidade do modelo.

$$
J(\theta) = -\underbrace{\frac{1}{n} \left[ \sum_{i=1}^{n} \sum_{j=0}^{k-1} \mathbf{1}[y^{(i)} = j] \log \frac{e^{\theta_j \cdot x^{(i)}/\tau}}{\sum_{l=0}^{k-1} e^{\theta_l \cdot x^{(i)}/\tau}} \right]}_{\text{Entropia Cruzada}} + \underbrace{\frac{\lambda}{2} \sum_{j=0}^{k-1} \sum_{s=0}^{d-1} \theta_{js}^2}_{\text{Regularização } L_2}
$$

onde:
- $ n $ é o número total de exemplos de treinamento,
- $ k $ é o número de classes distintas,
- $ d $ é o número de características por amostra,
- $ x^{(i)} $ representa o vetor de características da $ i $-ésima amostra,
- $ y^{(i)} $ é a classe verdadeira da $ i $-ésima amostra,
- $ \theta_j $ é o vetor de pesos para a classe $ j $,
- $ \tau $ é um **parâmetro de temperatura**, que pode ser usado para controlar a suavização da distribuição de probabilidade (comum em modelos de aprendizado profundo),
- $ \lambda $ é um hiperparâmetro de **regularização**.

### **Termo Principal: Log Loss (_Cross-Entropy_)**
O primeiro termo da função de custo é a **entropia cruzada (cross-entropy loss)**, amplamente utilizada para classificação multiclasse. Ele mede o quão bem o modelo atribui probabilidades corretas às classes reais.

$$
-\frac{1}{n} \sum_{i=1}^{n} \sum_{j=0}^{k-1} \mathbf{1}[y^{(i)} = j] \log P(y^{(i)} = j | x^{(i)})
$$

onde:

$$
P(y^{(i)} = j | x^{(i)}) = \frac{e^{\theta_j \cdot x^{(i)}/\tau}}{\sum_{l=0}^{k-1} e^{\theta_l \cdot x^{(i)}/\tau}}
$$

Essa fração representa a **função softmax**, que converte os valores de ativação linear ($ \theta \cdot x $) em probabilidades.

### **Termo de Regularização**
O segundo termo:

$$
\frac{\lambda}{2} \sum_{j=0}^{k-1} \sum_{s=0}^{d-1} \theta_{js}^2
$$

é uma **regularização L2 (Ridge Regularization)**, que penaliza grandes valores de $ \theta $ para evitar **overfitting**. Ele incentiva o modelo a aprender pesos menores e mais estáveis.

## **Relação com Regressão Logística Softmax**
A [**Regressão Logística Softmax**](https://github.com/SampMark/Machine-Learning/blob/main/Softmax_Multinomial_Regression.ipynb) estende a regressão logística binária para múltiplas classes. Para um conjunto de dados de classificação com $ k $ classes, a função softmax converte os escores do modelo em probabilidades:

$$
P(y=j | x, \theta) = \frac{e^{\theta_j \cdot x}}{\sum_{l=0}^{k-1} e^{\theta_l \cdot x}}
$$

Isso garante que a soma das probabilidades seja 1, permitindo que o modelo seja interpretado como um **classificador probabilístico**.


## **Comparação com Outras Funções de Custo**
1. **Erro Quadrático Médio (MSE)**:  
   - Funciona bem para **regressão**, mas não é ideal para **classificação**, pois trata classes como valores contínuos.
   
2. **Cross-Entropy Loss (para classificação binária)**:  
   - Similar à softmax, mas aplicada a **dois rótulos**.

3. [**Hinge Loss (SVMs)**](https://github.com/SampMark/Machine-Learning/blob/main/Hinge_Loss.ipynb):  
   - Em vez de probabilidades, trabalha com **margens de decisão**.

A função softmax é preferida para problemas **multiclasse**, já que garante que cada entrada seja classificada corretamente com uma probabilidade associada.

---


In [1]:
# @title Importando as bibliotecas
import numpy as np

## **🎯Implementação em Python**

---

In [6]:
# @title **Função de custo para a regressão logística Softmax com regularização L2**
def compute_cost_function(X: np.ndarray, Y: np.ndarray, theta: np.ndarray,
                          lambda_factor: float, temp_parameter: float = 1.0) -> float:
    """
    Calcula a função de custo para a regressão logística Softmax com regularização L2.

    Args:
        X (np.ndarray): Matriz de características com shape (n_amostras, n_features).
        Y (np.ndarray): Vetor de rótulos inteiros com shape (n_amostras,).
        theta (np.ndarray): Matriz de pesos do modelo com shape (n_classes, n_features).
        lambda_factor (float): Fator de regularização para penalização L2.
        temp_parameter (float, opcional): Parâmetro de temperatura para escalonamento. Padrão é 1.0.

    Returns:
        float: Valor da função de custo.
    """

    # Número de amostras e dimensões dos dados
    n, d = X.shape  # n = número de amostras, d = número de características
    k = theta.shape[0]  # Número de classes

    # Verificações das entradas
    assert Y.shape[0] == n, "Erro: Y deve ter o mesmo número de linhas que X."
    assert theta.shape[1] == d, "Erro: theta deve ter o mesmo número de colunas que X."
    assert isinstance(lambda_factor, float) and lambda_factor >= 0, "Erro: lambda_factor deve ser um número positivo."

    # Cálculo dos logits escalonados: θX / temp_parameter
    logits = np.dot(X, theta.T) / temp_parameter  # Shape: (n, k)

    # Estabilização numérica: subtrai o máximo dos logits para evitar overflow
    logits -= np.max(logits, axis=1, keepdims=True)

    # Aplicação da função Softmax
    exp_logits = np.exp(logits)
    softmax_probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # One-hot encoding dos rótulos
    y_onehot = np.eye(k)[Y.flatten()]  # Shape: (n, k)

    # Probabilidades das classes corretas
    correct_class_probs = np.sum(softmax_probs * y_onehot, axis=1)

    # Cálculo da entropia cruzada (evita log(0) com np.clip)
    cross_entropy = -np.mean(np.log(np.clip(correct_class_probs, 1e-10, 1.0)))

    # Regularização L2
    regularization = (lambda_factor / 2) * np.sum(theta ** 2)

    # Função de custo final
    cost = cross_entropy + regularization

    return cost

### **📌 Explicação do Código**

#### **1️⃣ Cálculo dos Logits**
```python
logits = np.dot(X, theta.T) / temp_parameter
```
- Multiplicação da matriz de características $X$ pelos pesos $\theta$, ajustados pelo **parâmetro de temperatura**.
- **Objetivo**: transformar as características em escores que representam a força da previsão para cada classe.

#### **2️⃣ Estabilização Numérica**
```python
logits -= np.max(logits, axis=1, keepdims=True)
```
- **Por quê?** Em aprendizado de máquina, os expoentes podem crescer muito, levando a `overflow numérico`, logo, subtrair o máximo evita esse problema.

#### **3️⃣ Aplicação da Softmax**
```python
exp_logits = np.exp(logits)
softmax_probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
```
- **Função Softmax** converte os logits em probabilidades entre 0 e 1.
- **Objetivo**: Garante que a soma das probabilidades para cada amostra seja 1.

#### **4️⃣ Criação do One-Hot Encoding**
```python
y_onehot = np.eye(k)[Y.flatten()]
```
- Gera uma matriz **one-hot encoded** para os rótulos de classe \( Y \).
- Se \( k=3 \) e \( Y=[0, 2, 1] \), então:
  ```python
  y_onehot = [[1, 0, 0],  # Classe 0
              [0, 0, 1],  # Classe 2
              [0, 1, 0]]  # Classe 1
  ```

#### **5️⃣ Cálculo da Entropia Cruzada**
```python
correct_class_probs = np.sum(softmax_probs * y_onehot, axis=1)
cross_entropy = -np.mean(np.log(np.clip(correct_class_probs, 1e-10, 1.0)))
```
- **Objetivo**: Penaliza predições erradas através do logaritmo das probabilidades corretas.
- **Por que `np.clip`?** Evita `log(0)`, que geraria `-inf`.

#### **6️⃣ Regularização L2**
```python
regularization = (lambda_factor / 2) * np.sum(theta ** 2)
```
- Penaliza grandes valores de $\theta$, evitando **overfitting**.

### **7️⃣ Cálculo da Função de Custo Final**
```python
cost = cross_entropy + regularization
```
- Soma a entropia cruzada e a penalização $L2$.

## **Exemplo 1 - Classificação de Dígitos (MNIST Simples)**


---


A seguir uma simulação simples num conjunto de dados de classificação de dígitos manuscritos (como MNIST), num cenário típico de um classificador de dígitos manuscritos com **10 classes (0 a 9)** e um conjunto de características aleatórias. A função de custo avalia a performance inicial do modelo antes do treinamento.

In [7]:
# @title **Simulação de um conjunto de dados MNIST**
np.random.seed(42)  # Para reprodutibilidade

n_samples = 1000   # Número de exemplos
n_features = 784   # Características (como pixels de uma imagem 28x28)
n_classes = 10     # Dígitos de 0 a 9

# Gera características aleatórias normalizadas
X = np.random.rand(n_samples, n_features)

# Gera rótulos aleatórios entre 0 e 9
Y = np.random.randint(0, n_classes, size=(n_samples,))

# Inicializa pesos aleatórios
theta = np.random.randn(n_classes, n_features) * 0.01

# Define hiperparâmetros
lambda_factor = 0.1  # Regularização
temp_parameter = 1.0  # Parâmetro de temperatura para softmax

# Cálculo do custo
cost = compute_cost_function(X, Y, theta, lambda_factor, temp_parameter)
print(f"Função de custo (MNIST Simulado): {cost:.4f}")


Função de custo (MNIST Simulado): 2.3604


## **Exemplo 2 - Classificação de Sentimentos/Opniões (Análise de Texto)**

---


Neste segundo exemplo, utilizamos um conjunto de dados **bag-of-words** para classificar sentimentos/opiniões extraídas de comentários (positivo, neutro, negativo). Uma aplicação da **função de custo em PLN** (Processamento de Linguagem Natural), em que matriz `X` representa um **modelo de Bag-of-Words** (frequência de palavras) e a função de custo fornece um indicativo de quão bem o modelo inicial está classificado os sentimentos/opiniões sobre produtos/serviços, entre outras aplicações.

In [8]:
# Simulação de um dataset de análise de sentimentos
np.random.seed(123)

n_samples = 500    # Número de frases analisadas
n_features = 300   # Número de características (palavras)
n_classes = 3      # Classes: [0 = negativo, 1 = neutro, 2 = positivo]

# Características de frequência de palavras
X = np.random.rand(n_samples, n_features)

# Rótulos para sentimentos (negativo, neutro, positivo)
Y = np.random.randint(0, n_classes, size=(n_samples,))

# Inicializa pesos do modelo
theta = np.random.randn(n_classes, n_features) * 0.01

# Parâmetros de treinamento
lambda_factor = 0.05
temp_parameter = 1.0

# Calculando o custo
cost = compute_cost_function(X, Y, theta, lambda_factor, temp_parameter)
print(f"Função de custo (Análise de Sentimentos): {cost:.4f}")

Função de custo (Análise de Sentimentos): 1.1054


## **Referências**

* GOODFELLOW, I.; BENGIO, Y.; COURVILLE, A. **Deep Learning**. Cambridge: MIT Press, 2016. Capítulo sobre redes neurais e otimização. Disponível em: [https://www.deeplearningbook.org/](https://www.deeplearningbook.org/). Acesso em: 26 de fev. 2025.

